# Embedding Dynamics

Tracks how token embeddings evolve through the network: identity decay, semantic drift, positional persistence, and context mixing.

**References:**
- Ethayarajh (2019) "How Contextual are Contextualized Word Representations?"
- Cai et al. (2021) "Isotropy in the Contextual Embedding Space"

## Why This Matters

Embedding dynamics tracks how token representations evolve across layers — identity decay (how quickly the original token embedding is overwritten), semantic drift, and context mixing. This reveals the transformation from token-level to contextual representations.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.embedding_dynamics import (
    token_identity_decay,
    semantic_drift_analysis,
    positional_encoding_persistence,
    embedding_subspace_tracking,
    context_mixing_rate,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")

In [ ]:
# 1. Token identity decay
decay = token_identity_decay(model, tokens, pos=-1)
print(f"Half-life layer: {decay['half_life_layer']}")
print(f"Decay rate: {decay['decay_rate']:.4f}")
print(f"Final identity: {decay['final_identity']:.4f}")
print(f"\nIdentity trajectory:")
for l, v in enumerate(decay['identity_trajectory']):
    bar = '#' * int(max(0, v) * 30)
    print(f"  Layer {l}: {v:+.4f} {bar}")

In [ ]:
# 2. Semantic drift analysis
drift = semantic_drift_analysis(model, tokens)
print(f"Total drift: {drift['total_drift']:.4f} radians")
print(f"Fastest drift layer: {drift['fastest_drift_layer']}")
for l in range(cfg.n_layers):
    print(f"  Layer {l}->{l+1}: {drift['drift_angles'][l]:.4f} rad (cumulative: {drift['cumulative_drift'][l]:.4f})")

In [ ]:
# 3. Positional encoding persistence
pos = positional_encoding_persistence(model, tokens)
print(f"Persistence score: {pos['persistence_score']:.4f}")
for l in range(len(pos['position_discriminability'])):
    order = 'yes' if pos['position_order_preserved'][l] else 'no'
    print(f"  Layer {l}: discrim={pos['position_discriminability'][l]:.4f}, distance={pos['mean_inter_position_distance'][l]:.4f}, order={order}")

In [ ]:
# 4. Embedding subspace tracking
sub = embedding_subspace_tracking(model, tokens, n_components=5)
print(f"Subspace exit layer: {sub['subspace_exit_layer']}")
print(f"Final subspace fraction: {sub['final_subspace_fraction']:.4f}")
for l in range(len(sub['embedding_subspace_projection'])):
    bar = '#' * int(sub['embedding_subspace_projection'][l] * 30)
    print(f"  Layer {l}: proj={sub['embedding_subspace_projection'][l]:.4f}, orth={sub['orthogonal_growth'][l]:.4f} {bar}")

In [ ]:
# 5. Context mixing rate
mix = context_mixing_rate(model, tokens, target_pos=-1)
print(f"Fastest mixing layer: {mix['fastest_mixing_layer']}")
for l in range(len(mix['self_similarity'])):
    print(f"  Layer {l}: self_sim={mix['self_similarity'][l]:.4f}, context_dep={mix['context_dependence'][l]:.4f}")